In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# 1. Muat Data
df_train = pd.read_csv('../data/processed/train.csv')
item_cols = ['item_id', 'title', 'release_date', 'video_release_date', 'imdb_url', 'unknown',
             'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary',
             'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance',
             'Sci-Fi', 'Thriller', 'War', 'Western']
df_items = pd.read_csv('../data/raw/ml-100k/u.item', sep='|', names=item_cols, encoding='latin-1')
genre_columns = item_cols[5:]
df_item_features = df_items[['item_id', 'title'] + genre_columns]

# 2. Rekonstruksi Matriks CF (SVD)
user_item_matrix = df_train.pivot(index='user_id', columns='item_id', values='rating').fillna(0)
svd = TruncatedSVD(n_components=20, random_state=42)
user_factors = svd.fit_transform(user_item_matrix)
item_factors = svd.components_
df_cf_preds = pd.DataFrame(np.dot(user_factors, item_factors), index=user_item_matrix.index, columns=user_item_matrix.columns)

# 3. Rekonstruksi Matriks CB (Cosine Similarity)
item_similarity = cosine_similarity(df_item_features[genre_columns].values)
df_item_sim = pd.DataFrame(item_similarity, index=df_item_features['item_id'], columns=df_item_features['item_id'])

print("Data CF dan CB berhasil direkonstruksi dan siap digabungkan!")

Data CF dan CB berhasil direkonstruksi dan siap digabungkan!


In [2]:
from sklearn.preprocessing import MinMaxScaler

def get_hybrid_recommendations(user_id, top_n=5, weight_cf=0.7, weight_cb=0.3):
    # 1. Ambil Skor CF
    if user_id not in df_cf_preds.index:
        return {"error": "User Cold-Start: Gunakan Popularity Baseline"}
    cf_scores = df_cf_preds.loc[user_id]
    
    # 2. Ambil Skor CB
    user_history = df_train[(df_train['user_id'] == user_id) & (df_train['rating'] >= 4)]
    if user_history.empty:
        return {"error": "Tidak ada histori positif untuk CB"}
    liked_item_ids = user_history['item_id'].tolist()
    cb_scores = df_item_sim.loc[liked_item_ids].mean(axis=0)
    
    # 3. Gabungkan dalam satu tabel untuk dinormalisasi
    df_scores = pd.DataFrame({'cf_score': cf_scores, 'cb_score': cb_scores}).fillna(0)
    
    # 4. Normalisasi (Wajib dilakukan agar skor 1-5 dan 0-1 bisa setara)
    scaler = MinMaxScaler()
    df_scores[['cf_score_norm', 'cb_score_norm']] = scaler.fit_transform(df_scores[['cf_score', 'cb_score']])
    
    # 5. Hitung Skor Hybrid (Weighted Sum)
    df_scores['hybrid_score'] = (df_scores['cf_score_norm'] * weight_cf) + (df_scores['cb_score_norm'] * weight_cb)
    
    # 6. Hapus item yang sudah pernah dilihat
    all_seen_items = df_train[df_train['user_id'] == user_id]['item_id'].tolist()
    df_scores = df_scores.drop(all_seen_items, errors='ignore')
    
    # 7. Ambil Top N
    top_items = df_scores.sort_values('hybrid_score', ascending=False).head(top_n)
    
    # 8. Format Output
    result = []
    for item_id, row in top_items.iterrows():
        title = df_item_features[df_item_features['item_id'] == item_id]['title'].values[0]
        result.append({
            "item_id": int(item_id),
            "title": title,
            "hybrid_score": round(row['hybrid_score'], 3),
            "cf_contribution": round(row['cf_score_norm'] * weight_cf, 3), # Porsi pengaruh CF
            "cb_contribution": round(row['cb_score_norm'] * weight_cb, 3)  # Porsi pengaruh CB
        })
        
    return {
        "user_id": user_id,
        "recommendation_type": "hybrid (cf+cb)",
        "recommendations": result
    }

# Kita tes dengan bobot 70% CF dan 30% CB untuk User 12
print("Rekomendasi Hybrid (70% CF, 30% CB) untuk User 12:")
for rec in get_hybrid_recommendations(user_id=12, top_n=5, weight_cf=0.7, weight_cb=0.3)['recommendations']:
    print(rec)

Rekomendasi Hybrid (70% CF, 30% CB) untuk User 12:
{'item_id': 64, 'title': 'Shawshank Redemption, The (1994)', 'hybrid_score': np.float64(0.759), 'cf_contribution': np.float64(0.511), 'cb_contribution': np.float64(0.249)}
{'item_id': 22, 'title': 'Braveheart (1995)', 'hybrid_score': np.float64(0.717), 'cf_contribution': np.float64(0.465), 'cb_contribution': np.float64(0.252)}
{'item_id': 423, 'title': 'E.T. the Extra-Terrestrial (1982)', 'hybrid_score': np.float64(0.704), 'cf_contribution': np.float64(0.542), 'cb_contribution': np.float64(0.162)}
{'item_id': 357, 'title': "One Flew Over the Cuckoo's Nest (1975)", 'hybrid_score': np.float64(0.684), 'cf_contribution': np.float64(0.435), 'cb_contribution': np.float64(0.249)}
{'item_id': 655, 'title': 'Stand by Me (1986)', 'hybrid_score': np.float64(0.681), 'cf_contribution': np.float64(0.42), 'cb_contribution': np.float64(0.261)}
